In [ ]:
import os
import re
import sys
import logging
import json
import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
import plotly.graph_objects as go
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from utils.db_interface import get_engine
from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----------------------
# Configuration
# ----------------------
MODELS_DIR = "../../../models/calibrated"
table_name = "merged_mix_features"
PRED_DIR ="../../../models/calibrated/preds"
SUBANALYSIS_DIR = "../results"
LOG_LEVEL = logging.INFO
# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
# -----
# DB connection
engine = get_engine()

)

## Helper functions

In [ ]:
def compute_bin_thresholds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    mid = (q1 + q3) / 2
    return {
        1: f"< {q1:.1f}",
        2: f"[{q1:.1f} - {mid:.1f})",
        3: f"[{mid:.1f} - {q3:.1f})",
        4: f">= {q3:.1f}"
    }


## Evaluate Subgroups 

In [ ]:
with open("test_full_dfs.pkl", "rb") as f:
    test_full_dfs = pickle.load(f)

In [ ]:
# ausgehend auf einem dataframe "test_full" (kommend entweder aus mix, inv oder noninv, eine Funktion die ein Feature auswählen lässt um eine Subgruppe zu definieren und die Modell Vorhersagen auf die Subgruppe evaluiert
def evaluate_subgroup(
    df: pd.DataFrame,
    filter_condition=None,
    label_col: str = "label",
    pred_col: str = "preds",
    threshold: float = 0.5,
    model_name: str = "Unnamed Model",  # Hinzugefügt
    name: str = "Unnamed Subgroup",
    verbose: bool = True
):
    """
    Evaluiert eine Teilmenge des Testsets anhand von ROC AUC und anderen Metriken.
    
    Parameters:
        df: Testset mit Label und Vorhersage
        filter_condition: bool-Series oder Lambda zum Filtern (z.B. df['treatment_count'] != 0)
        label_col: Name der Label-Spalte
        pred_col: Name der Prediction-Spalte
        threshold: Schwellenwert für die Binärisierung der Vorhersagen
        model_name: Name des Modells (für Log-Ausgabe)
        name: Name der Subgruppe (für Logging)
    """
    if filter_condition is None:
        subset = df
    elif callable(filter_condition):
        subset = df[filter_condition(df)]
    else:
        subset = df[filter_condition]

    if subset.empty:
        print(f"Subgroup '{name}' is empty.")
        return None

    y_true = subset[label_col]
    y_pred_prob = subset[pred_col]
    threshold = float(threshold)
    y_pred_label = (y_pred_prob >= threshold).astype(int)

    # Verteilung der Labels
    #positive_rate = y_true.mean()
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    n_total = n_pos + n_neg
    
    # Prüfe, ob pos_count zu klein ist ( n= 30?)
    # if n_pos < 30:
    #     print(f"Subgroup '{name}' has too few positive samples ({n_pos}). Skipping evaluation.")
    #     return None

    auc = roc_auc_score(y_true, y_pred_prob)
    cm = confusion_matrix(y_true, y_pred_label)

    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')

    if verbose:
        print(f"Subgroup: {name} | Model: {model_name} | Threshold: {threshold}")
        print(f"  → N = {len(subset)}")
        print(f"  → AUC = {auc:.3f}")
        print(f"  → Sensitivity (TPR) = {sensitivity:.3f}")
        print(f"  → Specificity (TNR) = {specificity:.3f}")
    #    print(f"  → Confusion Matrix:\n{cm}\n")

    return {
    "auc": auc,
    "n": len(subset),
    "n_pos": int(n_pos),
    #"n_neg": int(n_neg),
    "n_total": int(n_total),
    "sensitivity": sensitivity,
    "specificity": specificity,
     "threshold": threshold
}


In [ ]:
#wrapper!

def evaluate_all_subgroups(
    dfs: dict,  # Dictionary mit allen DataFrames für die Modelle (z. B. 'mix_tg', 'inv_ntg', ...)
    threshold_dict: dict,  # Dictionary mit Thresholds für jedes Modell
    features: list,
    label_col: str = "label",
    pred_col: str = "preds",
    min_samples: int = 30,  # Optional: Skip zu kleine Gruppen
    verbose: bool = True
):
    """
    Evaluates model performance for each subgroup of each given feature in all datasets.
    
    Parameters:
        dfs: Dictionary of DataFrames for each model variant (e.g. 'mix_tg', 'inv_ntg', ...)
        threshold_dict: Dictionary containing thresholds for each model variant
        features: List of column names to stratify on
        label_col: Name of the label column
        pred_col: Name of the predicted probability column
        min_samples: Skip subgroups with fewer samples than this
    """
    results = []

    # Iteriere über alle Modelle/Datasets im `dfs`-Dictionary
    for model_name, df in dfs.items():
        # Wähle den passenden Threshold für das Modell aus dem Dictionary
        threshold = threshold_dict.get(model_name)  # Falls kein Threshold für das Modell vorhanden ist, soll es einen Fehler werfen
        if threshold is None:
            raise ValueError(f"Threshold für Modell '{model_name}' nicht gefunden.")

        # Evaluierung für jedes Feature in den Daten
        for feature in features:
            unique_vals = sorted(df[feature].dropna().unique())

            for val in unique_vals:
                name = f"{feature} == {val}"
                condition = df[feature] == val

                if condition.sum() < min_samples:
                    if verbose:
                        print(f"Skipping subgroup '{name}' (n={condition.sum()}) - too small.")
                    continue

                # Rufe evaluate_subgroup für jede Subgruppe im DataFrame auf
                result = evaluate_subgroup(
                    df=df,
                    filter_condition=condition,
                    label_col=label_col,
                    pred_col=pred_col,
                    threshold=threshold,  # Übergabe des Schwellenwerts
                    name=name,
                    verbose=verbose
                )
                if result:
                    result['model_name'] = model_name
                    result['feature'] = feature
                    result['value'] = val
                    results.append(result)

    # Rückgabe der gesammelten Ergebnisse
    return pd.DataFrame(results)

## Bau der Subgroupanalysis table

In [ ]:
def evaluate_all_subgroups_latex_ready(
    dfs: dict,
    features: list,  # List of column names to stratify on
    threshold_dict: dict,
    label_col: str = "label",
    pred_col: str = "preds",
    min_samples: int = 30,
    verbose: bool = True
):
    rows = []

  # Lade die Binning-Information einmalig am Anfang
    with open("/utils/thresholds_all_types.json", "r") as f:
        binning_dict = json.load(f)

    for feature in features:
        subgroup_rows = defaultdict(dict)

        for model_name, df in dfs.items():
            # Wähle den passenden Threshold für das Modell aus dem Dictionary
            threshold = threshold_dict.get(model_name)  # Falls kein Threshold für das Modell vorhanden ist, soll es einen Fehler werfen
            if verbose:
                print("threshold", threshold)
                print(type(threshold))
            
            # Prüfe, ob das Feature in der DataFrame-Spalte vorhanden ist
            if feature not in df.columns:
                if verbose:
                    print(f"Skipping {feature} for {model_name} (not in DataFrame)")
                continue

            unique_vals = sorted(df[feature].dropna().unique())

            for val in unique_vals:
                condition = df[feature] == val
                if condition.sum() < min_samples:
                    if verbose:
                        print(f"Skipping {feature}={val} for {model_name} (n too small: {condition.sum()})")
                    continue

                if verbose:
                    print(f"Evaluating {feature}={val} for {model_name}...")

                    print(f"Model: {model_name}, Feature: {feature}, Value: {val}, Value Type: {type(val)}")
                    print(df[feature].dtype)
                
                result = evaluate_subgroup(
                    df=df,
                    filter_condition=condition,
                    label_col=label_col,
                    pred_col=pred_col,
                    model_name=model_name,
                    threshold=threshold,  # Übergabe des Schwellenwerts
                    name=f"{feature} == {val}",
                    verbose=verbose
                )

                

                if result is None:
                    continue

                auc = f"{result['auc']:.3f}"
                count_str = f"{result['n_pos']}/{result['n_total']}"
                sens = f"{result['sensitivity']:.3f}" if 'sensitivity' in result else "N/A"
                spec = f"{result['specificity']:.3f}" if 'specificity' in result else "N/A"
                threshold = f"{result['threshold']:.6f}" if 'threshold' in result else "N/A"
                entry = f"{auc},{count_str}, {sens}, {spec}, {threshold}"
                # Füge die Ergebnisse für diese Subgruppe und Modell in das Dictionary ein

                subgroup_rows[(feature, val)][model_name] = entry

        # Füge die Ergebnisse zeilenweise zusammen
        for (feature, val), model_entries in subgroup_rows.items():
            row = {
                "Feature": f"{feature} = {val}"
            }
            for model_name in dfs.keys():
                row[model_name] = model_entries.get(model_name, "")
            rows.append(row)

    return pd.DataFrame(rows)



In [ ]:
# AUfruf von compute bin_threhsolds (e.g. Age_bin1 = <53)
types = ["mix", "inv", "noninv"]
thresholds_all_types = {}

for type_ in types:
    subjects_query = f"""
    SELECT
    {type_}.subject_id,
    mv.age,
    mv.height,
    mv.weight,
    mv.bmi FROM ce_approach.merged_{type_}_features {type_}
    JOIN ce_approach.mv_metadata mv ON {type_}.subject_id = mv.subject_id;""" 

    engine = get_engine()
    df_subj = pd.read_sql(subjects_query, engine)

    # Beispiel für mehrere Merkmale
    thresholds_all_types[type_] = {
        "age": compute_bin_thresholds(df_subj["age"]),
        "height": compute_bin_thresholds(df_subj["height"]),
        "weight": compute_bin_thresholds(df_subj["weight"]),
        "bmi": compute_bin_thresholds(df_subj["bmi"])
    }

#with open("/nfs/work/abax6050/hypotension/src/richards_workspace/hypotension-individual-threshold/ce_approach/utils/thresholds_all_types.json", "w") as f:
#    json.dump(thresholds_all_types, f)


In [ ]:
##### AUFRUF VON evaluate_all_subgroups_latex_ready ######
# Dictionary mit DataFrames für jedes Modell
dfs_ntg = {
    "mix_ntg": test_full_dfs["mix_ntg"],
    "inv_ntg": test_full_dfs["inv_ntg"],
    "noninv_ntg": test_full_dfs["noninv_ntg"]
}
#from Noah, 0.80:
decision_thresholds = {
    "mix_ntg": 0.0011452179169282317,
    "inv_ntg": 0.0020985996816307306,
    "noninv_ntg": 0.0008126477478072047
}

# Liste der Modelltypen
types = ["mix", "inv", "noninv"]

# Features, auf denen stratifiziert werden soll
features = [
     'gender_bin',
       'ethnicity_bin', 'age_bin', 'height_bin', 'weight_bin', 'bmi_bin',
       'obesity', 'hypertension', 'diabetes', 'kidney_disease', 'lung_disease',
       'heart_disease', 'drug_abuse', 'depression', 'sedatives_given',
       'blood_products_transfusions_given', 'antibiotics_given',
       'anticoagulants_antiplatelets_given', 'neuromuscular_blockers_given',
       'analgesics_given', 'crystalloids_given', 'electrolytes_given',
       'gi_protection_given', 'parenteral_nutrition_given',
       'antiarrhythmics_given', 'positive_sample',
       'positive_event', 'map_elevation_range', 'hypotension_risk_factor'
]

# Für Ergebnisse
all_results = []

results = evaluate_all_subgroups_latex_ready(
    dfs=dfs_ntg,
    features=features,
    threshold_dict=decision_thresholds,
    verbose=False
)
    
#print(type(results))

results.to_csv(os.path.join(SUBANALYSIS_DIR, "results_subgroup_analysis_newest_2.csv"), index=False)
